In [ ]:
# 安装需要依赖的库
%pip install trl
%pip install peft

In [ ]:
# 设置环境变量
%env HF_ENDPOINT=https://hf-mirror.com
%env HF_HOME=/root/autodl-tmp/hf
%env TENSORBOARD_LOGGING_DIR=/root/tf-logs

In [ ]:
# 加载model和tkennizer
from transformers import AutoModelForCausalLM,AutoTokenizer

# 模型名称
model_name = "Qwen/Qwen3-8B"

model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
# 数据集
from datasets import load_dataset

dataset_dict = load_dataset('json', data_files = {"train":"datasets/keywords_data_train.jsonl", "test":"datasets/keywords_data_test.jsonl"})

def map_func(example):
    conversation = example['conversation']
    messages = []
    for item in conversation:
        messages.append({'role':'user','content':item['human']})
        messages.append({'role':'assistant','content':item['assistant']})
    return {'messages':messages}

dataset_dict = dataset_dict.map(map_func,batched=False,remove_columns=['dataset','conversation','category','conversation_id'])

In [ ]:
# 微调参数
from trl import SFTConfig,SFTTrainer

from peft import LoraConfig

# Configure LoRA parameters
# r: rank dimension for LoRA update matrices (smaller = more compression)
rank_dimension = 6
# lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
lora_alpha = 12
# lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)
lora_dropout = 0.05

peft_config = LoraConfig(
    r=rank_dimension,  # Rank dimension - typically between 4-32
    lora_alpha=lora_alpha,  # LoRA scaling factor - typically 2x rank
    lora_dropout=lora_dropout,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA. the corresponding biases will be updated during training.
    target_modules="all-linear",  # Which modules to apply LoRA to
    task_type="CAUSAL_LM",  # Task type for model architecture
)

# Configure trainer
training_args = SFTConfig(
    output_dir="/root/autodl-tmp/sft/Qwen3-8B/LoRA",
    max_steps=1000,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    bf16=True,
    warmup_steps=50
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["test"],
    processing_class=tokenizer,
    peft_config=peft_config
)

In [ ]:
# 开始训练

trainer.train()

In [ ]:
# 保存最优模型
trainer.save_model('/root/autodl-tmp/sft/Qwen3-8B/LoRA/best')